In [2]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI



# Connecting to OpenAI

Load in the environment variables in your `.env` file and connect to OpenAI.  

In [4]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [5]:
openai = OpenAI()



# Let's make a quick call to a OpenAI model 

In [23]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! Can you tell me a joke about programming?"
response = openai.chat.completions.create(model="gpt-4o-mini", messages=[{"role":"user", "content":message}])
print(response.choices[0].message.content)

Sure! Here's one for you:

Why do programmers prefer dark mode?

Because light attracts bugs!


## OK onwards with our first project

In [7]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [25]:
# Let's try one out. Change the website and add print statements to follow along.

techcrunch = Website("https://techcrunch.com/")
print(techcrunch.title)
print(techcrunch.text)

TechCrunch | Startup and Technology News
–:–:–:–
Buy One Founder Summit Ticket, Get a Second for Free.
Register now.
Disrupt doors open Oct. 13. Get your pass and bring someone with you at 50% off.
REGISTER NOW.
Close
TechCrunch Desktop Logo
TechCrunch Mobile Logo
Latest
Startups
Venture
Apple
Security
AI
Apps
Disrupt 2026
Events
Podcasts
Newsletters
Search
Submit
Site Search Toggle
Mega Menu Toggle
Topics
Latest
AI
Amazon
Apps
Biotech & Health
Climate
Cloud Computing
Commerce
Crypto
Enterprise
EVs
Fintech
Fundraising
Gadgets
Gaming
Google
Government & Policy
Hardware
Instagram
Layoffs
Media & Entertainment
Meta
Microsoft
Privacy
Robotics
Security
Social
Space
Startups
TikTok
Transportation
Venture
More from TechCrunch
Staff
Events
Startup Battlefield
StrictlyVC
Newsletters
Podcasts
Videos
Partner Content
TechCrunch Brand Studio
Contact Us
Image Credits:
TechCrunch
Space
SpaceX and Tesla boards form special committees to review merger
Kirsten Korosec
10 hours ago
AI
SpaceX’s AI busines

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [9]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes website content and provides a concise summary,\
                 ignoring navigation-related text and irrelevant UI elements. Respond in Markdown."

In [10]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [26]:
print(user_prompt_for(techcrunch))

You are looking at a website titled TechCrunch | Startup and Technology News
The contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.

–:–:–:–
Buy One Founder Summit Ticket, Get a Second for Free.
Register now.
Disrupt doors open Oct. 13. Get your pass and bring someone with you at 50% off.
REGISTER NOW.
Close
TechCrunch Desktop Logo
TechCrunch Mobile Logo
Latest
Startups
Venture
Apple
Security
AI
Apps
Disrupt 2026
Events
Podcasts
Newsletters
Search
Submit
Site Search Toggle
Mega Menu Toggle
Topics
Latest
AI
Amazon
Apps
Biotech & Health
Climate
Cloud Computing
Commerce
Crypto
Enterprise
EVs
Fintech
Fundraising
Gadgets
Gaming
Google
Government & Policy
Hardware
Instagram
Layoffs
Media & Entertainment
Meta
Microsoft
Privacy
Robotics
Security
Social
Space
Startups
TikTok
Transportation
Venture
More from TechCrunch
Staff
Events
Startup Battlefield
StrictlyVC
Newsletters
Podcasts
V

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```


## And now let's build useful messages for GPT-4o-mini, using a function

In [14]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [27]:
# Try this out, and then try for a few more websites

messages_for(techcrunch)

[{'role': 'system',
  'content': 'You are an assistant that analyzes website content and provides a concise summary,                 ignoring navigation-related text and irrelevant UI elements. Respond in Markdown.'},
 {'role': 'user',
  'content': "You are looking at a website titled TechCrunch | Startup and Technology News\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\n–:–:–:–\nBuy One Founder Summit Ticket, Get a Second for Free.\nRegister now.\nDisrupt doors open Oct. 13. Get your pass and bring someone with you at 50% off.\nREGISTER NOW.\nClose\nTechCrunch Desktop Logo\nTechCrunch Mobile Logo\nLatest\nStartups\nVenture\nApple\nSecurity\nAI\nApps\nDisrupt 2026\nEvents\nPodcasts\nNewsletters\nSearch\nSubmit\nSite Search Toggle\nMega Menu Toggle\nTopics\nLatest\nAI\nAmazon\nApps\nBiotech & Health\nClimate\nCloud Computing\nCommerce\nCrypto\nEnterprise\nEVs\nFin

## Time to bring it together - the API for OpenAI is very simple!

In [28]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [31]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [32]:
display_summary("https://techcrunch.com/")

# TechCrunch Overview

TechCrunch is a leading technology news website that focuses on startups and technological advances. It covers a wide array of topics including AI, security, apps, enterprise technology, and venture capital.

## Key Highlights:
- **Current Promotions**: Buy one ticket for the Founder Summit and receive a second ticket for free.
- **Events**: TechCrunch Disrupt 2026 is set to open doors on October 13, 2026.

## Recent News:
1. **AI Innovations**:
   - **Plants Communication**: A startup has developed an AI model to interpret how plants communicate using electric signals.
   - **Google vs Nvidia**: Google has entered a competitive landscape against Nvidia, targeting AI application developers.
   - **Anthropic**: The company is facing challenges with its cloud partnerships while proposing a new AI safety bill.

2. **SpaceX Developments**:
   - Both SpaceX and Tesla's boards are forming committees to evaluate a potential merger.

3. **OpenAI Acquisitions**:
   - OpenAI has acquired Orchestra Bio, aiming to enhance its biotech sector.

4. **Transportation Insights**:
   - Google is integrating a personalized version of its Gemini AI in vehicles, while Rivian plans to implement AI agents in its cars.

5. **Company News**:
   - General Motors will close its Detroit battery assembly facility for unspecified reasons.

## Notes:
- The site features podcasts, newsletters, and articles aimed at keeping readers informed about significant trends and developments in the technology and startup sectors. 
- Recent articles discuss themes such as the increasing integration of AI in various industries and the challenges startups face in navigating partnerships and funding.

# Let's try more websites

In [33]:
display_summary("https://cnn.com")

# CNN Website Summary

CNN is a prominent news organization that offers breaking news, analysis, and video content across a wide range of topics. The website covers various categories such as U.S. news, world news, politics, business, health, entertainment, sports, science, and climate.

## Key News Highlights
- **Ukraine-Russia War:** Ongoing tensions and an impasse are reported, with a focus on diplomatic negotiations.
- **Israel-Hamas Conflict:** An increase in settler violence and recent arrests have been highlighted amid U.S. concerns.
- **France Protests:** Riots in France are analyzed as part of a larger public crisis, drawing parallels to potential issues within the U.S.
- **Drone Strike in Abyei:** A recent drone strike in the disputed area has resulted in a significant loss of life.
- **Health Reports:** New findings on cancer treatment predictions and Christina Applegate's revelation about a misdiagnosis are featured.
- **Post-Midterms Politics:** Discussions around Trump's rhetoric and internal GOP dynamics as the midterms approach are prominent.

The website also features interactive content, videos, and user engagement opportunities, allowing viewers to stay updated on the latest global news and participate in discussions.

In [34]:
display_summary("https://anthropic.com")

# Summary of Anthropic Website

Anthropic is a public benefit corporation focused on AI research and product development with a commitment to prioritizing safety and mitigating risks associated with AI. The organization aims to secure the benefits of AI while navigating its vast potential impact on society.

## Announcements
1. **Claude Opus 5.5** (Oct 7, 2026): This is the newest Opus model, noted for being faster and cost-efficient, requiring 40% less to operate.
   
2. **Claude Haiku 5.5** (Oct 7, 2026): The fastest and most capable small model designed for high-volume tasks such as summarization and browsing.

3. **Claude Sonnet 5.5** (Sep 28, 2026): An upgrade over Claude Sonnet 5, it offers improvements in speed (over 30% faster) and costs.

4. **Claude Opus 5.5** (Sep 22, 2026): This model excels in coding and knowledge work, and is also 40% cheaper to run compared to its predecessor, Opus 5.

The website includes resources regarding AI safety, alignment practices, and the organization’s responsible scaling policy, which addresses potential catastrophic risks from advanced AI systems. Additionally, the Anthropic Institute focuses on tackling the significant societal challenges posed by powerful AI.